# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    branch = input("GitHub 브랜치명 (기본: main): ").strip() or "main"
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--branch", branch, clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")
        subprocess.run(["git", "fetch", "origin", branch], cwd=repo_dir, check=True)
        subprocess.run(["git", "checkout", branch], cwd=repo_dir, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", branch], cwd=repo_dir, check=True)

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")


In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 저장된 tokenizer를 load합니다.
# Step 1을 이미 완료했다면 Google Drive/로컬 JSON을 그대로 재사용합니다.
try:
    from bpe import BPETokenizer

    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        tokenizer_candidates = [
            Path("/content/drive/MyDrive/gpt-lab/bpe_tokenizer_vocab3000.json"),
            Path("/content/drive/MyDrive/bpe_tokenizer_vocab3000.json"),
        ]
        tokenizer_save_path = tokenizer_candidates[0]
    else:
        tokenizer_candidates = [
            repo_dir / "artifacts" / "bpe_tokenizer_vocab3000.json",
            Path.home() / "Downloads" / "bpe_tokenizer_vocab3000.json",
        ]
        tokenizer_save_path = tokenizer_candidates[0]

    tokenizer = BPETokenizer(vocab_size=10000)
    tokenizer_path = next((path for path in tokenizer_candidates if path.exists()), None)
    if tokenizer_path is not None:
        tokenizer.load(tokenizer_path)
        print("BPE tokenizer loaded:", tokenizer_path)
    else:
        tokenizer_save_path.parent.mkdir(parents=True, exist_ok=True)
        print("BPE tokenizer not found. Training and saving:", tokenizer_save_path)
        tokenizer.train(corpus)
        tokenizer.save(tokenizer_save_path)

    vocab_size = len(tokenizer.id_to_token)
    print("vocab size:", vocab_size)

    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=vocab_size, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NameError:
    print("먼저 Step 1 tokenizer load 셀을 실행하세요.")
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": vocab_size,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NameError:
    print("먼저 Step 1 tokenizer load 셀을 실행하세요.")
except NotImplementedError as e:
    print("Model TODO 미구현:", e)


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": vocab_size,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NameError:
    print("먼저 Step 1 tokenizer load 셀을 실행하세요.")
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)


### Step 5 실제 사전 학습 실행

저장된 BPE tokenizer와 NSMC LM corpus로 작은 GPT를 실제로 학습합니다. 기본 설정은 Colab에서 빠르게 확인하는 용도이며, 시간이 더 있으면 `TRAIN_CHARS`, `NUM_EPOCHS`, 모델 크기를 키우세요.


In [ ]:
# 실제 pretraining용 dataloader/model/optimizer를 준비합니다.
# tokenization 결과는 Drive/artifacts에 캐시해서 런타임 재시작 후에도 바로 재사용합니다.
try:
    import torch
    from dataset import create_dataloader
    from model import GPTModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    # 빠른 확인:  TRAIN_CHARS=100_000,  VAL_CHARS=20_000,  NUM_EPOCHS=3
    # 적당한 학습: TRAIN_CHARS=500_000,  VAL_CHARS=50_000,  NUM_EPOCHS=5
    # 더 제대로:   TRAIN_CHARS=1_500_000, VAL_CHARS=100_000, NUM_EPOCHS=8
    # 오래 학습:   TRAIN_CHARS=len(corpus), VAL_CHARS=len(val_corpus), NUM_EPOCHS=10 이상
    CONTEXT_LENGTH = 64
    BATCH_SIZE = 16 if device.type == "cuda" else 4
    TRAIN_CHARS = min(len(corpus), 100_000)
    VAL_CHARS = min(len(val_corpus), 20_000)
    NUM_EPOCHS = 3

    if "google.colab" in sys.modules:
        cache_dir = Path("/content/drive/MyDrive/gpt-lab/cache")
    else:
        cache_dir = repo_dir / "artifacts" / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    token_cache_path = cache_dir / (
        f"pretrain_tokens_vocab{vocab_size}_ctx{CONTEXT_LENGTH}_"
        f"train{TRAIN_CHARS}_val{VAL_CHARS}.pt"
    )

    if token_cache_path.exists():
        cached = torch.load(token_cache_path, map_location="cpu")
        train_token_ids = cached["train_token_ids"]
        val_token_ids = cached["val_token_ids"]
        print("token cache loaded:", token_cache_path)
    else:
        print("token cache not found. Encoding corpus now:", token_cache_path)
        train_token_ids = tokenizer.encode(corpus[:TRAIN_CHARS])
        val_token_ids = tokenizer.encode(val_corpus[:VAL_CHARS])
        torch.save(
            {
                "train_token_ids": train_token_ids,
                "val_token_ids": val_token_ids,
                "vocab_size": vocab_size,
                "context_length": CONTEXT_LENGTH,
                "train_chars": TRAIN_CHARS,
                "val_chars": VAL_CHARS,
            },
            token_cache_path,
        )
        print("token cache saved:", token_cache_path)

    train_loader = create_dataloader(
        train_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=True,
        drop_last=True,
    )
    val_loader = create_dataloader(
        val_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=False,
    )

    pretrain_config = {
        "vocab_size": vocab_size,
        "context_length": CONTEXT_LENGTH,
        "emb_dim": 128,
        "n_heads": 4,
        "n_layers": 2,
        "drop_rate": 0.1,
        "qkv_bias": False,
    }
    model = GPTModel(pretrain_config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

    print("train batches:", len(train_loader), "val batches:", len(val_loader))
    print("train tokens:", len(train_token_ids), "val tokens:", len(val_token_ids))
except NameError:
    print("먼저 데이터 준비 셀과 Step 1 tokenizer load 셀을 실행하세요.")
except NotImplementedError as e:
    print("앞 단계 TODO 미구현:", e)


In [ ]:
# train_model을 실제로 실행하고 epoch별 train/validation loss를 기록합니다.
try:
    from train import calc_loss_loader, generate_and_print_sample, train_model

    EVAL_ITER = min(20, len(val_loader))
    EVAL_FREQ = max(1, len(train_loader) // 2)
    START_CONTEXT = "이 영화는"

    train_loss_history = []
    val_loss_history = []
    global_step = 0

    initial_train_loss = calc_loss_loader(train_loader, model, device, EVAL_ITER)
    initial_val_loss = calc_loss_loader(val_loader, model, device, EVAL_ITER)
    print(f"Before training: train loss {initial_train_loss:.3f}, val loss {initial_val_loss:.3f}")

    for epoch in range(NUM_EPOCHS):
        epoch_train_losses = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            num_epochs=epoch + 1,
            eval_freq=EVAL_FREQ,
            eval_iter=EVAL_ITER,
            start_context=START_CONTEXT,
            tokenizer=tokenizer,
            ckpt_freq=None,
            start_epoch=epoch,
            global_step=global_step,
        )
        global_step += len(train_loader)
        train_loss_history.extend(epoch_train_losses)

        val_loss = calc_loss_loader(val_loader, model, device, EVAL_ITER)
        val_loss_history.append(val_loss)
        print(f"Epoch {epoch + 1}: train loss {train_loss_history[-1]:.3f}, val loss {val_loss:.3f}")

    final_test_iter = min(100, len(val_loader))
    final_test_loss = calc_loss_loader(val_loader, model, device, final_test_iter)
    print(f"Final held-out validation loss ({final_test_iter} batches): {final_test_loss:.3f}")

    print("Final sample:")
    generate_and_print_sample(
        model=model,
        tokenizer=tokenizer,
        device=device,
        start_context=START_CONTEXT,
        max_new_tokens=80,
        context_size=CONTEXT_LENGTH,
        temperature=0.8,
        top_k=40,
    )
except NameError:
    print("먼저 pretraining 준비 셀을 실행하세요.")
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)


In [ ]:
# 학습 곡선을 그립니다. val loss가 train loss보다 살짝 높고 둘 다 내려가면 정상적인 신호입니다.
try:
    import matplotlib.pyplot as plt

    epochs = list(range(1, len(train_loss_history) + 1))
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, train_loss_history, marker="o", label="Train loss")
    plt.plot(epochs, val_loss_history, marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Mini GPT pretraining loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("train loss history:", [round(x, 3) for x in train_loss_history])
    print("val loss history:", [round(x, 3) for x in val_loss_history])
except NameError:
    print("먼저 실제 pretraining 실행 셀을 실행하세요.")


## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")